In [ ]:
# ==========================================
# Prediction Notebook: Energy Adoption Modeling
# ==========================================

In [ ]:
!pip uninstall -y xgboost
!pip install xgboost

In [ ]:
pip install scikit-learn xgboost autogluon

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
#from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, RocCurveDisplay

from autogluon.tabular import TabularPredictor

In [ ]:
df = pd.read_csv("/Users/macbookpro/Desktop/DKU/STATS 201/PS1/upgrade-e datasets/UPGRADE-E_V2.csv")
df

In [ ]:
df['IECC21'].unique()

In [ ]:
df['IECC21'].iloc[3]

In [ ]:
# create target variable for solar adoption
solar = ["mod_rene_wob", "mod_rene_wib", "mod_rene_sts", "mod_rene_wob_landlord", "mod_rene_wib_landlord", "mod_rene_sts_landlord"]
# Create new column: 1 if any adoption type is 1, else 0
df["solar_adoption"] = (df[solar].sum(axis=1) > 0).astype(int)

print(df["solar_adoption"].value_counts())

In [ ]:
# Data cleaning

# Clean income
df['hhincome'] = df['hhincome'].replace({'999': np.nan, '0': np.nan})

# Clean education
df['education'] = df['education'].replace({'999': np.nan, 'Prefer not to say': np.nan})

# Encode income
income_order = {
    '1-15,000': 1,
    '15,001-30,000': 2,
    '30,001-45,000': 3,
    '45,001-60,000': 4,
    '60,001-75,000': 5,
    '75,001-100,000': 6,
    '100,001-125,000': 7,
    '125,001-150,000': 8,
    '150,001-175,000': 9,
    '175,001 or more': 10
}

df['hhincome_num'] = df['hhincome'].map(income_order)

# Encode education
edu_order = {
    'Some high school, no diploma': 1,
    'High school diploma or GED': 2,
    'Associates degree or trade school': 3,
    "Bachelor's degree": 4,
    'Graduate or professional degree': 5
}

df['education_num'] = df['education'].map(edu_order)

# Encode own_rent
df['own_rent_num'] = df['own_rent'].map({'Rent': 0, 'Own': 1})

# Encode hometype
df = pd.get_dummies(df, columns=['hometype'], prefix='hometype')

# Encode region
df = pd.get_dummies(df, columns=['region'], prefix='region')

# Encode IECC21
df['IECC21'] = df['IECC21'].astype('Int64')

In [ ]:
# Select features and target

# Select base numeric/ordinal features
base_features = ["hhincome_num", "education_num", "own_rent_num", 
                 "race_white", "race_asian", "race_blackorafricanamerican", 
                 "IECC21"]

# Find all one-hot encoded columns for hometype and region
hometype_features = [col for col in df.columns if col.startswith("hometype_")]
region_features = [col for col in df.columns if col.startswith("region_")]

# Combine all features
features = base_features + hometype_features + region_features
target = "solar_adoption"

In [ ]:
# Drop missing rows (or impute if desired)
df = df[features + [target]].dropna()

X = df[features]
y = df[target]

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# --- Step 2: Logistic Regression ---
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [ ]:
# --- Step 3: Random Forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [ ]:
# --- Step 4: Gradient Boosting (XGBoost) ---
xgb = XGBClassifier(eval_metric="logloss", use_label_encoder=False)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

In [ ]:
# --- Step 5: Evaluation ---
def evaluate(model_name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba[:,1])
    print(f"{model_name} - Accuracy: {acc:.3f}, F1: {f1:.3f}, AUC: {auc:.3f}")
    return acc, f1, auc

metrics = []
metrics.append(evaluate("Logistic Regression", y_test, y_pred_lr, lr.predict_proba(X_test)))
metrics.append(evaluate("Random Forest", y_test, y_pred_rf, rf.predict_proba(X_test)))
metrics.append(evaluate("XGBoost", y_test, y_pred_xgb, xgb.predict_proba(X_test)))

In [ ]:
# --- Step 6: AutoGluon Benchmark ---
train_data, test_data = train_test_split(df, test_size=0.3, random_state=42)
predictor = TabularPredictor(label=target).fit(train_data)
results = predictor.evaluate(test_data)

In [ ]:
# --- Step 7: ROC Curves ---
RocCurveDisplay.from_estimator(lr, X_test, y_test)
plt.title("ROC Curve - Logistic Regression")
plt.show()

RocCurveDisplay.from_estimator(rf, X_test, y_test)
plt.title("ROC Curve - Random Forest")
plt.show()

RocCurveDisplay.from_estimator(xgb, X_test, y_test)
plt.title("ROC Curve - XGBoost")
plt.show()